In [ ]:
# =====================================================================
# STEP 1: Install Dependencies
# =====================================================================
!pip install -q earthengine-api pandas fastkml shapely

import ee
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import files
from fastkml import kml
from shapely.geometry import mapping

# =====================================================================
# STEP 2: Authenticate and Initialize Earth Engine
# =====================================================================
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    # Replace with your GCP Project ID
    ee.Initialize(project='ee-birukgetaneh13')

# =====================================================================
# STEP 3: Upload and Parse KML File
# =====================================================================
print("Please upload your KML file:")
uploaded = files.upload()

kml_filename = list(uploaded.keys())[0]
kml_data = uploaded[kml_filename]

def extract_geometries(element):
    geoms = []
    # Check if element is a collection/has nested features
    if hasattr(element, 'features'):
        for feat in element.features:
            geoms.extend(extract_geometries(feat))

    # Extract from Document, Folder, or Placemark containers
    if hasattr(element, 'geometry') and element.geometry is not None:
        geoms.append(element.geometry)
    return geoms

# Try standard fastkml parsing first
raw_geoms = []
try:
    k_obj = kml.KML()
    k_obj.from_string(kml_data)
    raw_geoms = extract_geometries(k_obj)
except Exception:
    pass

# Fallback to direct XML parsing if fastkml fails to find geometries
if not raw_geoms:
    try:
        root = ET.fromstring(kml_data)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        coords_elements = root.findall('.//kml:coordinates', namespaces)

        # If namespace query fails, try without namespace
        if not coords_elements:
            coords_elements = root.findall('.//coordinates')

        for elem in coords_elements:
            coords_text = elem.text.strip()
            coords_list = []
            for coord_str in coords_text.split():
                parts = coord_str.split(',')
                if len(parts) >= 2:
                    coords_list.append([float(parts[0]), float(parts[1])])
            if coords_list:
                # If coordinates close on themselves, make it a polygon
                if coords_list[0] == coords_list[-1] and len(coords_list) >= 4:
                    raw_geoms.append(ee.Geometry.Polygon([coords_list]))
                else:
                    raw_geoms.append(ee.Geometry.LineString(coords_list))
    except Exception as e:
        print(f"XML parsing fallback encountered error: {e}")

if not raw_geoms:
    raise ValueError("No geometries could be parsed from the uploaded KML file. Please ensure it contains active polygon boundaries.")

# Convert geometries safely
if isinstance(raw_geoms[0], ee.Geometry):
    ee_geometries = raw_geoms
else:
    ee_geometries = [ee.Geometry(mapping(g)) for g in raw_geoms]

# Define the region of interest
if len(ee_geometries) > 1:
    kml_roi = ee.Geometry.MultiPolygon([g.coordinates() for g in ee_geometries])
else:
    kml_roi = ee_geometries[0]

roi_feature = ee.FeatureCollection([ee.Feature(kml_roi)])
print("KML boundary loaded successfully.")

# =====================================================================
# STEP 4: Define Dates & Load Datasets for Wheat Season
# =====================================================================
# Wheat Crop Season Dates
planting_date = '2025-12-24'
harvesting_date = '2026-06-05'

datasets = {
    'AETI': {'id': 'projects/UNFAO/wapor/v3/L2-AETI-D', 'band': 'L2-AETI-D', 'scale': 0.1},
    'NPP':  {'id': 'projects/UNFAO/wapor/v3/L2-NPP-D',  'band': 'L2-NPP-D',  'scale': 0.001},
    'T':    {'id': 'projects/UNFAO/wapor/v3/L2-T-D',    'band': 'L2-T-D',    'scale': 0.1},
    'ETo':  {'id': 'projects/UNFAO/wapor/v3/L1-RET-D',  'band': 'L1-RET-D',  'scale': 0.1}
}

# Function to load collection and multiply daily rate by dekad day count
def load_and_scale_dekads(cfg, var_name):
    def process_img(img):
        # Calculate duration of the dekad in days
        start_time = ee.Date(img.get('system:time_start'))
        end_time = ee.Date(img.get('system:time_end'))
        n_days = end_time.difference(start_time, 'day')

        # Multiply daily rate by dekad days to get dekadal total
        daily_rate = img.select(cfg['band']).multiply(cfg['scale'])
        # Cast explicitly to float to avoid mismatched type conflicts across the collection
        dekad_total = daily_rate.multiply(n_days).rename(var_name).toFloat()
        return dekad_total

    return (ee.ImageCollection(cfg['id'])
            .filterDate(planting_date, harvesting_date)
            .filterBounds(kml_roi)
            .map(process_img))

aeti_col = load_and_scale_dekads(datasets['AETI'], 'AETI_mm')
npp_col  = load_and_scale_dekads(datasets['NPP'],  'NPP_gC_m2')
t_col    = load_and_scale_dekads(datasets['T'],    'T_mm')
eto_col  = load_and_scale_dekads(datasets['ETo'],  'ETo_mm')

# =====================================================================
# STEP 5: Seasonal Aggregation & Biomass Conversion
# =====================================================================
# Sum across all dekads in the wheat growing period
seasonal_aeti = aeti_col.sum()
seasonal_npp  = npp_col.sum()
seasonal_t    = t_col.sum()
seasonal_eto  = eto_col.sum()

# Convert NPP sum to Total Biomass (Biomass = 14.443 * NPP)
seasonal_biomass = seasonal_npp.multiply(14.443).rename('Total_Biomass')

# Combine all aggregated bands into a single composite raster
coords = ee.Image.pixelLonLat()
seasonal_composite = ee.Image.cat([
    seasonal_aeti,
    seasonal_eto,
    seasonal_t,
    seasonal_npp,
    seasonal_biomass,
    coords
])

# =====================================================================
# STEP 6: Sample Pixel Values across KML Boundary
# =====================================================================
sampled_pixels = seasonal_composite.sampleRegions(
    collection=roi_feature,
    scale=100,
    geometries=False
)

def format_pixel_data(feature):
    lon = feature.get('longitude')
    lat = feature.get('latitude')
    pixel_id = ee.String('Pixel_').cat(ee.Number(lon).format('%.5f')).cat('_').cat(ee.Number(lat).format('%.5f'))
    return feature.set({'pixel_id': pixel_id})

final_features = sampled_pixels.map(format_pixel_data)

print("Extracting seasonal pixel metrics from Earth Engine...")
raw_data = final_features.getInfo()

# =====================================================================
# STEP 7: Structure Data & Save CSV
# =====================================================================
parsed_records = []
for f in raw_data['features']:
    props = f['properties']
    parsed_records.append({
        'Pixel_ID': props.get('pixel_id'),
        'Longitude': props.get('longitude'),
        'Latitude': props.get('latitude'),
        'Crop': 'Wheat',
        'Planting_Date': planting_date,
        'Harvesting_Date': harvesting_date,
        'Seasonal_AETI_mm': props.get('AETI_mm'),
        'Seasonal_ETo_mm': props.get('ETo_mm'),
        'Seasonal_Transpiration_mm': props.get('T_mm'),
        'Seasonal_NPP_gC_m2': props.get('NPP_gC_m2'),
        'Total_Biomass': props.get('Total_Biomass')
    })

df = pd.DataFrame(parsed_records)

# Clean up any missing values
df = df.dropna(subset=['Total_Biomass'])

csv_filename = "wheat_season_wapor_aggregated_pixels.csv"
df.to_csv(csv_filename, index=False)
print(f"Extraction complete! Saved {len(df)} pixel records to {csv_filename}")

files.download(csv_filename)

Please upload your KML file:


Saving chefana irrigation.kml to chefana irrigation (6).kml
KML boundary loaded successfully.
Extracting seasonal pixel metrics from Earth Engine...
Extraction complete! Saved 23 pixel records to wheat_season_wapor_aggregated_pixels.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Load the saved CSV to guarantee the statistics match the exported file
export_df = pd.read_csv(csv_filename)

# Select the numeric columns for statistics
numeric_cols = [
    'Seasonal_AETI_mm',
    'Seasonal_ETo_mm',
    'Seasonal_Transpiration_mm',
    'Seasonal_NPP_gC_m2',
    'Total_Biomass'
]

# Generate descriptive statistics
summary_stats = export_df[numeric_cols].describe()
display(summary_stats)

,Seasonal_AETI_mm,Seasonal_ETo_mm,Seasonal_Transpiration_mm,Seasonal_NPP_gC_m2,Total_Biomass
count,23.000000,23.000000,23.000000,23.000000,23.000000
mean,385.608695,607.499994,317.513042,543.403522,7848.377062
std,69.200584,0.000000,70.865156,97.140249,1402.996621
min,181.900001,607.499994,107.000000,236.305996,3412.967499
25%,368.650001,607.499994,301.900000,528.310002,7630.381357
50%,405.699998,607.499994,336.899997,565.778004,8171.531707
75%,426.699997,607.499994,359.700001,602.379502,8700.167152
max,465.099998,607.499994,398.099998,654.044003,9446.357529


In [ ]:
# =====================================================================
# STEP 1: Install Dependencies & Initialize Earth Engine
# =====================================================================
!pip install -q earthengine-api pandas fastkml shapely

import ee
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import files
from fastkml import kml
from shapely.geometry import mapping

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-birukgetaneh13')
except Exception:
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project='ee-birukgetaneh13')

# Upload & Parse KML
print("Please upload your KML file:")
uploaded = files.upload()

kml_filename = list(uploaded.keys())[0]
kml_data = uploaded[kml_filename]

def extract_geometries(element):
    geoms = []
    if hasattr(element, 'features'):
        for feat in element.features:
            geoms.extend(extract_geometries(feat))
    if hasattr(element, 'geometry') and element.geometry is not None:
        geoms.append(element.geometry)
    return geoms

# Try standard fastkml parsing first
raw_geoms = []
try:
    k = kml.KML()
    k.from_string(kml_data)
    raw_geoms = extract_geometries(k)
except Exception:
    pass

# Fallback to direct XML parsing if fastkml fails to find geometries
if not raw_geoms:
    try:
        root = ET.fromstring(kml_data)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        coords_elements = root.findall('.//kml:coordinates', namespaces)
        if not coords_elements:
            coords_elements = root.findall('.//coordinates')

        for elem in coords_elements:
            coords_text = elem.text.strip()
            coords_list = []
            for coord_str in coords_text.split():
                parts = coord_str.split(',')
                if len(parts) >= 2:
                    coords_list.append([float(parts[0]), float(parts[1])])
            if coords_list:
                if coords_list[0] == coords_list[-1] and len(coords_list) >= 4:
                    raw_geoms.append(ee.Geometry.Polygon([coords_list]))
                else:
                    raw_geoms.append(ee.Geometry.LineString(coords_list))
    except Exception as e:
        print(f"XML parsing fallback encountered error: {e}")

if not raw_geoms:
    raise ValueError("No geometries could be parsed from the uploaded KML file.")

# Convert geometries safely
if isinstance(raw_geoms[0], ee.Geometry):
    ee_geometries = raw_geoms
else:
    ee_geometries = [ee.Geometry(mapping(g)) for g in raw_geoms]

kml_roi = ee.Geometry.MultiPolygon([g.coordinates() for g in ee_geometries]) if len(ee_geometries) > 1 else ee_geometries[0]
roi_feature = ee.FeatureCollection([ee.Feature(kml_roi)])
print("KML boundary loaded successfully.")

# =====================================================================
# STEP 2: Define Dates & Load Datasets
# =====================================================================
planting_date = '2025-12-24'
harvesting_date = '2026-06-05'

# 1. FAO WaPOR v3 AETI (Sum across season in mm)
aeti_col = (ee.ImageCollection("projects/UNFAO/wapor/v3/L2-AETI-D")
            .filterDate(planting_date, harvesting_date)
            .filterBounds(kml_roi)
            .map(lambda img: img.select('L2-AETI-D').multiply(0.1)
                 .multiply(ee.Date(img.get('system:time_end')).difference(ee.Date(img.get('system:time_start')), 'day')).toFloat()))
seasonal_aeti = aeti_col.sum().rename('Seasonal_AETI_mm').clip(kml_roi)

# 2. Sentinel-2 Peak/Maximum NDVI (10m Resolution)
def add_ndvi(img):
    return img.addBands(img.normalizedDifference(['B8', 'B4']).rename('NDVI'))

s2_ndvi = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(planting_date, harvesting_date)
           .filterBounds(kml_roi)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .map(add_ndvi)
           .select('NDVI')
           .max()  # Captures peak growing season vigor
           .rename('Peak_NDVI')
           .clip(kml_roi))

# 3. Sentinel-1 Volumetric Soil Moisture Conversion (10m Resolution)
s1_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(kml_roi)
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV'))

# Historical reference range for backscatter normalization (last 3 years)
s1_min = s1_col.filterDate('2023-01-01', '2026-06-05').min() # Dry backscatter state
s1_max = s1_col.filterDate('2023-01-01', '2026-06-05').max() # Wet backscatter state

# Current season mean Sentinel-1 VV backscatter
s1_seasonal = s1_col.filterDate(planting_date, harvesting_date).mean()

# Normalized relative index (0 to 1 scale)
ssm_relative = (s1_seasonal.subtract(s1_min)) \
    .divide(s1_max.subtract(s1_min)) \
    .clamp(0, 1)

# Physical Volumetric Soil Moisture (VSM) conversion
# Adjust vsm_min and vsm_max based on your regional soil texture (e.g., Loam/Silt Loam bounds)
vsm_min = 0.10  # 10% Volumetric Water Content (Dry boundary)
vsm_max = 0.42  # 42% Volumetric Water Content (Wet / Field Capacity boundary)

vsm = ssm_relative.multiply(vsm_max - vsm_min).add(vsm_min) \
    .multiply(100) \
    .rename('Volumetric_Soil_Moisture_Pct') \
    .clip(kml_roi)

# =====================================================================
# STEP 3: Combine & Sample Pixel Values for Export
# =====================================================================
coords = ee.Image.pixelLonLat()
combined_composite = ee.Image.cat([
    seasonal_aeti,
    s2_ndvi,
    vsm,
    coords
])

sampled_pixels = combined_composite.sampleRegions(
    collection=roi_feature,
    scale=100,  # Matches WaPOR grid scale
    geometries=False
)

def format_pixel_data(feature):
    lon = feature.get('longitude')
    lat = feature.get('latitude')
    pixel_id = ee.String('Pixel_').cat(ee.Number(lon).format('%.5f')).cat('_').cat(ee.Number(lat).format('%.5f'))
    return feature.set({'pixel_id': pixel_id})

final_features = sampled_pixels.map(format_pixel_data)

print("Extracting pixel metrics from Earth Engine...")
raw_data = final_features.getInfo()

parsed_records = []
for f in raw_data['features']:
    props = f['properties']
    parsed_records.append({
        'Pixel_ID': props.get('pixel_id'),
        'Longitude': props.get('longitude'),
        'Latitude': props.get('latitude'),
        'Seasonal_AETI_mm': props.get('Seasonal_AETI_mm'),
        'Peak_NDVI': props.get('Peak_NDVI'),
        'Volumetric_Soil_Moisture_Pct': props.get('Volumetric_Soil_Moisture_Pct')
    })

df = pd.DataFrame(parsed_records).dropna(subset=['Seasonal_AETI_mm'])

csv_filename = "wapor_aeti_peakndvi_vsm_pixels.csv"
df.to_csv(csv_filename, index=False)
print(f"Complete! Downloaded {len(df)} pixel records to {csv_filename}")

files.download(csv_filename)

Please upload your KML file:


Saving Chacha cluster1_A=24 ha.kml to Chacha cluster1_A=24 ha.kml
KML boundary loaded successfully.
Extracting pixel metrics from Earth Engine...
Complete! Downloaded 23 pixel records to wapor_aeti_peakndvi_vsm_pixels.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================================================================
# STEP 1: Install Dependencies
# =====================================================================
!pip install -q earthengine-api pandas fastkml shapely

import ee
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import files
from fastkml import kml
from shapely.geometry import mapping

# =====================================================================
# STEP 2: Authenticate and Initialize Earth Engine
# =====================================================================
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    # Replace with your GCP Project ID
    ee.Initialize(project='ee-birukgetaneh13')

# =====================================================================
# STEP 3: Upload and Parse KML File
# =====================================================================
print("Please upload your KML file:")
uploaded = files.upload()

kml_filename = list(uploaded.keys())[0]
kml_data = uploaded[kml_filename]

def extract_geometries(element):
    geoms = []
    # Check if element is a collection/has nested features
    if hasattr(element, 'features'):
        for feat in element.features:
            geoms.extend(extract_geometries(feat))

    # Extract from Document, Folder, or Placemark containers
    if hasattr(element, 'geometry') and element.geometry is not None:
        geoms.append(element.geometry)
    return geoms

# Try standard fastkml parsing first
raw_geoms = []
try:
    k_obj = kml.KML()
    k_obj.from_string(kml_data)
    raw_geoms = extract_geometries(k_obj)
except Exception:
    pass

# Fallback to direct XML parsing if fastkml fails to find geometries
if not raw_geoms:
    try:
        root = ET.fromstring(kml_data)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        coords_elements = root.findall('.//kml:coordinates', namespaces)

        # If namespace query fails, try without namespace
        if not coords_elements:
            coords_elements = root.findall('.//coordinates')

        for elem in coords_elements:
            coords_text = elem.text.strip()
            coords_list = []
            for coord_str in coords_text.split():
                parts = coord_str.split(',')
                if len(parts) >= 2:
                    coords_list.append([float(parts[0]), float(parts[1])])
            if coords_list:
                # If coordinates close on themselves, make it a polygon
                if coords_list[0] == coords_list[-1] and len(coords_list) >= 4:
                    raw_geoms.append(ee.Geometry.Polygon([coords_list]))
                else:
                    raw_geoms.append(ee.Geometry.LineString(coords_list))
    except Exception as e:
        print(f"XML parsing fallback encountered error: {e}")

if not raw_geoms:
    raise ValueError("No geometries could be parsed from the uploaded KML file. Please ensure it contains active polygon boundaries.")

# Convert geometries safely
if isinstance(raw_geoms[0], ee.Geometry):
    ee_geometries = raw_geoms
else:
    ee_geometries = [ee.Geometry(mapping(g)) for g in raw_geoms]

# Define the region of interest
if len(ee_geometries) > 1:
    kml_roi = ee.Geometry.MultiPolygon([g.coordinates() for g in ee_geometries])
else:
    kml_roi = ee_geometries[0]

roi_feature = ee.FeatureCollection([ee.Feature(kml_roi)])
print("KML boundary loaded successfully.")

# =====================================================================
# STEP 4: Define Dates & Load Datasets for Wheat Season
# =====================================================================
# Wheat Crop Season Dates
planting_date = '2025-12-24'
harvesting_date = '2026-06-05'

datasets = {
    'AETI': {'id': 'projects/UNFAO/wapor/v3/L2-AETI-D', 'band': 'L2-AETI-D', 'scale': 0.1},
    'NPP':  {'id': 'projects/UNFAO/wapor/v3/L2-NPP-D',  'band': 'L2-NPP-D',  'scale': 0.001},
    'T':    {'id': 'projects/UNFAO/wapor/v3/L2-T-D',    'band': 'L2-T-D',    'scale': 0.1},
    'ETo':  {'id': 'projects/UNFAO/wapor/v3/L1-RET-D',  'band': 'L1-RET-D',  'scale': 0.1}
}

# Function to load collection and multiply daily rate by dekad day count
def load_and_scale_dekads(cfg, var_name):
    def process_img(img):
        # Calculate duration of the dekad in days
        start_time = ee.Date(img.get('system:time_start'))
        end_time = ee.Date(img.get('system:time_end'))
        n_days = end_time.difference(start_time, 'day')

        # Multiply daily rate by dekad days to get dekadal total
        daily_rate = img.select(cfg['band']).multiply(cfg['scale'])
        # Cast explicitly to float to avoid mismatched type conflicts across the collection
        dekad_total = daily_rate.multiply(n_days).rename(var_name).toFloat()
        return dekad_total

    return (ee.ImageCollection(cfg['id'])
            .filterDate(planting_date, harvesting_date)
            .filterBounds(kml_roi)
            .map(process_img))

aeti_col = load_and_scale_dekads(datasets['AETI'], 'AETI_mm')
npp_col  = load_and_scale_dekads(datasets['NPP'],  'NPP_gC_m2')
t_col    = load_and_scale_dekads(datasets['T'],    'T_mm')
eto_col  = load_and_scale_dekads(datasets['ETo'],  'ETo_mm')

# =====================================================================
# STEP 5: Seasonal Aggregation & Biomass Conversion
# =====================================================================
# Sum across all dekads in the wheat growing period
seasonal_aeti = aeti_col.sum()
seasonal_npp  = npp_col.sum()
seasonal_t    = t_col.sum()
seasonal_eto  = eto_col.sum()

# Convert NPP sum to Total Biomass (Biomass = 14.443 * NPP)
seasonal_biomass = seasonal_npp.multiply(14.443).rename('Total_Biomass')

# Combine all aggregated bands into a single composite raster
coords = ee.Image.pixelLonLat()
seasonal_composite = ee.Image.cat([
    seasonal_aeti,
    seasonal_eto,
    seasonal_t,
    seasonal_npp,
    seasonal_biomass,
    coords
])

# =====================================================================
# STEP 6: Sample Pixel Values across KML Boundary
# =====================================================================
sampled_pixels = seasonal_composite.sampleRegions(
    collection=roi_feature,
    scale=100,
    geometries=False
)

def format_pixel_data(feature):
    lon = feature.get('longitude')
    lat = feature.get('latitude')
    pixel_id = ee.String('Pixel_').cat(ee.Number(lon).format('%.5f')).cat('_').cat(ee.Number(lat).format('%.5f'))
    return feature.set({'pixel_id': pixel_id})

final_features = sampled_pixels.map(format_pixel_data)

print("Extracting seasonal pixel metrics from Earth Engine...")
raw_data = final_features.getInfo()

# =====================================================================
# STEP 7: Structure Data & Save CSV
# =====================================================================
parsed_records = []
for f in raw_data['features']:
    props = f['properties']
    parsed_records.append({
        'Pixel_ID': props.get('pixel_id'),
        'Longitude': props.get('longitude'),
        'Latitude': props.get('latitude'),
        'Crop': 'Wheat',
        'Planting_Date': planting_date,
        'Harvesting_Date': harvesting_date,
        'Seasonal_AETI_mm': props.get('AETI_mm'),
        'Seasonal_ETo_mm': props.get('ETo_mm'),
        'Seasonal_Transpiration_mm': props.get('T_mm'),
        'Seasonal_NPP_gC_m2': props.get('NPP_gC_m2'),
        'Total_Biomass': props.get('Total_Biomass')
    })

df = pd.DataFrame(parsed_records)

# Clean up any missing values
df = df.dropna(subset=['Total_Biomass'])

csv_filename = "wheat_season_wapor_aggregated_pixels.csv"
df.to_csv(csv_filename, index=False)
print(f"Extraction complete! Saved {len(df)} pixel records to {csv_filename}")

files.download(csv_filename)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 2.4 MB/s eta 0:00:00
Please upload your KML file:


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Saving Chacha cluster2_A=86 ha.kml to Chacha cluster2_A=86 ha.kml
KML boundary loaded successfully.
Extracting seasonal pixel metrics from Earth Engine...
Extraction complete! Saved 89 pixel records to wheat_season_wapor_aggregated_pixels.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================================================================
# STEP 1: Install Dependencies
# =====================================================================
!pip install -q earthengine-api pandas fastkml shapely

import ee
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import files
from fastkml import kml
from shapely.geometry import mapping

# =====================================================================
# STEP 2: Authenticate and Initialize Earth Engine
# =====================================================================
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    # Replace with your GCP Project ID
    ee.Initialize(project='ee-birukgetaneh13')

# =====================================================================
# STEP 3: Upload and Parse KML File
# =====================================================================
print("Please upload your KML file:")
uploaded = files.upload()

kml_filename = list(uploaded.keys())[0]
kml_data = uploaded[kml_filename]

def extract_geometries(element):
    geoms = []
    # Check if element is a collection/has nested features
    if hasattr(element, 'features'):
        for feat in element.features:
            geoms.extend(extract_geometries(feat))

    # Extract from Document, Folder, or Placemark containers
    if hasattr(element, 'geometry') and element.geometry is not None:
        geoms.append(element.geometry)
    return geoms

# Try standard fastkml parsing first
raw_geoms = []
try:
    k_obj = kml.KML()
    k_obj.from_string(kml_data)
    raw_geoms = extract_geometries(k_obj)
except Exception:
    pass

# Fallback to direct XML parsing if fastkml fails to find geometries
if not raw_geoms:
    try:
        root = ET.fromstring(kml_data)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        coords_elements = root.findall('.//kml:coordinates', namespaces)

        # If namespace query fails, try without namespace
        if not coords_elements:
            coords_elements = root.findall('.//coordinates')

        for elem in coords_elements:
            coords_text = elem.text.strip()
            coords_list = []
            for coord_str in coords_text.split():
                parts = coord_str.split(',')
                if len(parts) >= 2:
                    coords_list.append([float(parts[0]), float(parts[1])])
            if coords_list:
                # If coordinates close on themselves, make it a polygon
                if coords_list[0] == coords_list[-1] and len(coords_list) >= 4:
                    raw_geoms.append(ee.Geometry.Polygon([coords_list]))
                else:
                    raw_geoms.append(ee.Geometry.LineString(coords_list))
    except Exception as e:
        print(f"XML parsing fallback encountered error: {e}")

if not raw_geoms:
    raise ValueError("No geometries could be parsed from the uploaded KML file. Please ensure it contains active polygon boundaries.")

# Convert geometries safely
if isinstance(raw_geoms[0], ee.Geometry):
    ee_geometries = raw_geoms
else:
    ee_geometries = [ee.Geometry(mapping(g)) for g in raw_geoms]

# Define the region of interest
if len(ee_geometries) > 1:
    kml_roi = ee.Geometry.MultiPolygon([g.coordinates() for g in ee_geometries])
else:
    kml_roi = ee_geometries[0]

roi_feature = ee.FeatureCollection([ee.Feature(kml_roi)])
print("KML boundary loaded successfully.")

# =====================================================================
# STEP 4: Define Dates & Load Datasets for Wheat Season
# =====================================================================
# Wheat Crop Season Dates
planting_date = '2025-12-24'
harvesting_date = '2026-06-05'

datasets = {
    'AETI': {'id': 'projects/UNFAO/wapor/v3/L2-AETI-D', 'band': 'L2-AETI-D', 'scale': 0.1},
    'NPP':  {'id': 'projects/UNFAO/wapor/v3/L2-NPP-D',  'band': 'L2-NPP-D',  'scale': 0.001},
    'T':    {'id': 'projects/UNFAO/wapor/v3/L2-T-D',    'band': 'L2-T-D',    'scale': 0.1},
    'ETo':  {'id': 'projects/UNFAO/wapor/v3/L1-RET-D',  'band': 'L1-RET-D',  'scale': 0.1}
}

# Function to load collection and multiply daily rate by dekad day count
def load_and_scale_dekads(cfg, var_name):
    def process_img(img):
        # Calculate duration of the dekad in days
        start_time = ee.Date(img.get('system:time_start'))
        end_time = ee.Date(img.get('system:time_end'))
        n_days = end_time.difference(start_time, 'day')

        # Multiply daily rate by dekad days to get dekadal total
        daily_rate = img.select(cfg['band']).multiply(cfg['scale'])
        # Cast explicitly to float to avoid mismatched type conflicts across the collection
        dekad_total = daily_rate.multiply(n_days).rename(var_name).toFloat()
        return dekad_total

    return (ee.ImageCollection(cfg['id'])
            .filterDate(planting_date, harvesting_date)
            .filterBounds(kml_roi)
            .map(process_img))

aeti_col = load_and_scale_dekads(datasets['AETI'], 'AETI_mm')
npp_col  = load_and_scale_dekads(datasets['NPP'],  'NPP_gC_m2')
t_col    = load_and_scale_dekads(datasets['T'],    'T_mm')
eto_col  = load_and_scale_dekads(datasets['ETo'],  'ETo_mm')

# =====================================================================
# STEP 5: Seasonal Aggregation & Biomass Conversion
# =====================================================================
# Sum across all dekads in the wheat growing period
seasonal_aeti = aeti_col.sum()
seasonal_npp  = npp_col.sum()
seasonal_t    = t_col.sum()
seasonal_eto  = eto_col.sum()

# Convert NPP sum to Total Biomass (Biomass = 14.443 * NPP)
seasonal_biomass = seasonal_npp.multiply(14.443).rename('Total_Biomass')

# Combine all aggregated bands into a single composite raster
coords = ee.Image.pixelLonLat()
seasonal_composite = ee.Image.cat([
    seasonal_aeti,
    seasonal_eto,
    seasonal_t,
    seasonal_npp,
    seasonal_biomass,
    coords
])

# =====================================================================
# STEP 6: Sample Pixel Values across KML Boundary
# =====================================================================
sampled_pixels = seasonal_composite.sampleRegions(
    collection=roi_feature,
    scale=100,
    geometries=False
)

def format_pixel_data(feature):
    lon = feature.get('longitude')
    lat = feature.get('latitude')
    pixel_id = ee.String('Pixel_').cat(ee.Number(lon).format('%.5f')).cat('_').cat(ee.Number(lat).format('%.5f'))
    return feature.set({'pixel_id': pixel_id})

final_features = sampled_pixels.map(format_pixel_data)

print("Extracting seasonal pixel metrics from Earth Engine...")
raw_data = final_features.getInfo()

# =====================================================================
# STEP 7: Structure Data & Save CSV
# =====================================================================
parsed_records = []
for f in raw_data['features']:
    props = f['properties']
    parsed_records.append({
        'Pixel_ID': props.get('pixel_id'),
        'Longitude': props.get('longitude'),
        'Latitude': props.get('latitude'),
        'Crop': 'Wheat',
        'Planting_Date': planting_date,
        'Harvesting_Date': harvesting_date,
        'Seasonal_AETI_mm': props.get('AETI_mm'),
        'Seasonal_ETo_mm': props.get('ETo_mm'),
        'Seasonal_Transpiration_mm': props.get('T_mm'),
        'Seasonal_NPP_gC_m2': props.get('NPP_gC_m2'),
        'Total_Biomass': props.get('Total_Biomass')
    })

df = pd.DataFrame(parsed_records)

# Clean up any missing values
df = df.dropna(subset=['Total_Biomass'])

csv_filename = "wheat_season_wapor_aggregated_pixels.csv"
df.to_csv(csv_filename, index=False)
print(f"Extraction complete! Saved {len(df)} pixel records to {csv_filename}")

files.download(csv_filename)

Please upload your KML file:


Saving Chacha cluster3_A=31 ha.kml to Chacha cluster3_A=31 ha.kml
KML boundary loaded successfully.
Extracting seasonal pixel metrics from Earth Engine...
Extraction complete! Saved 32 pixel records to wheat_season_wapor_aggregated_pixels.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================================================================
# STEP 1: Install Dependencies & Initialize Earth Engine
# =====================================================================
!pip install -q earthengine-api pandas fastkml shapely

import ee
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import files
from fastkml import kml
from shapely.geometry import mapping

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-birukgetaneh13')
except Exception:
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project='ee-birukgetaneh13')

# Upload & Parse KML
print("Please upload your KML file:")
uploaded = files.upload()

kml_filename = list(uploaded.keys())[0]
kml_data = uploaded[kml_filename]

def extract_geometries(element):
    geoms = []
    if hasattr(element, 'features'):
        for feat in element.features:
            geoms.extend(extract_geometries(feat))
    if hasattr(element, 'geometry') and element.geometry is not None:
        geoms.append(element.geometry)
    return geoms

# Try standard fastkml parsing first
raw_geoms = []
try:
    k = kml.KML()
    k.from_string(kml_data)
    raw_geoms = extract_geometries(k)
except Exception:
    pass

# Fallback to direct XML parsing if fastkml fails to find geometries
if not raw_geoms:
    try:
        root = ET.fromstring(kml_data)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        coords_elements = root.findall('.//kml:coordinates', namespaces)
        if not coords_elements:
            coords_elements = root.findall('.//coordinates')

        for elem in coords_elements:
            coords_text = elem.text.strip()
            coords_list = []
            for coord_str in coords_text.split():
                parts = coord_str.split(',')
                if len(parts) >= 2:
                    coords_list.append([float(parts[0]), float(parts[1])])
            if coords_list:
                if coords_list[0] == coords_list[-1] and len(coords_list) >= 4:
                    raw_geoms.append(ee.Geometry.Polygon([coords_list]))
                else:
                    raw_geoms.append(ee.Geometry.LineString(coords_list))
    except Exception as e:
        print(f"XML parsing fallback encountered error: {e}")

if not raw_geoms:
    raise ValueError("No geometries could be parsed from the uploaded KML file.")

# Convert geometries safely
if isinstance(raw_geoms[0], ee.Geometry):
    ee_geometries = raw_geoms
else:
    ee_geometries = [ee.Geometry(mapping(g)) for g in raw_geoms]

kml_roi = ee.Geometry.MultiPolygon([g.coordinates() for g in ee_geometries]) if len(ee_geometries) > 1 else ee_geometries[0]
roi_feature = ee.FeatureCollection([ee.Feature(kml_roi)])
print("KML boundary loaded successfully.")

# =====================================================================
# STEP 2: Define Dates & Load Datasets
# =====================================================================
planting_date = '2025-12-24'
harvesting_date = '2026-06-05'

# 1. FAO WaPOR v3 AETI (Sum across season in mm)
aeti_col = (ee.ImageCollection("projects/UNFAO/wapor/v3/L2-AETI-D")
            .filterDate(planting_date, harvesting_date)
            .filterBounds(kml_roi)
            .map(lambda img: img.select('L2-AETI-D').multiply(0.1)
                 .multiply(ee.Date(img.get('system:time_end')).difference(ee.Date(img.get('system:time_start')), 'day')).toFloat()))
seasonal_aeti = aeti_col.sum().rename('Seasonal_AETI_mm').clip(kml_roi)

# 2. Sentinel-2 Peak/Maximum NDVI (10m Resolution)
def add_ndvi(img):
    return img.addBands(img.normalizedDifference(['B8', 'B4']).rename('NDVI'))

s2_ndvi = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(planting_date, harvesting_date)
           .filterBounds(kml_roi)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .map(add_ndvi)
           .select('NDVI')
           .max()  # Captures peak growing season vigor
           .rename('Peak_NDVI')
           .clip(kml_roi))

# 3. Sentinel-1 Volumetric Soil Moisture Conversion (10m Resolution)
s1_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(kml_roi)
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV'))

# Historical reference range for backscatter normalization (last 3 years)
s1_min = s1_col.filterDate('2023-01-01', '2026-06-05').min() # Dry backscatter state
s1_max = s1_col.filterDate('2023-01-01', '2026-06-05').max() # Wet backscatter state

# Current season mean Sentinel-1 VV backscatter
s1_seasonal = s1_col.filterDate(planting_date, harvesting_date).mean()

# Normalized relative index (0 to 1 scale)
ssm_relative = (s1_seasonal.subtract(s1_min)) \
    .divide(s1_max.subtract(s1_min)) \
    .clamp(0, 1)

# Physical Volumetric Soil Moisture (VSM) conversion
# Adjust vsm_min and vsm_max based on your regional soil texture (e.g., Loam/Silt Loam bounds)
vsm_min = 0.10  # 10% Volumetric Water Content (Dry boundary)
vsm_max = 0.42  # 42% Volumetric Water Content (Wet / Field Capacity boundary)

vsm = ssm_relative.multiply(vsm_max - vsm_min).add(vsm_min) \
    .multiply(100) \
    .rename('Volumetric_Soil_Moisture_Pct') \
    .clip(kml_roi)

# =====================================================================
# STEP 3: Combine & Sample Pixel Values for Export
# =====================================================================
coords = ee.Image.pixelLonLat()
combined_composite = ee.Image.cat([
    seasonal_aeti,
    s2_ndvi,
    vsm,
    coords
])

sampled_pixels = combined_composite.sampleRegions(
    collection=roi_feature,
    scale=100,  # Matches WaPOR grid scale
    geometries=False
)

def format_pixel_data(feature):
    lon = feature.get('longitude')
    lat = feature.get('latitude')
    pixel_id = ee.String('Pixel_').cat(ee.Number(lon).format('%.5f')).cat('_').cat(ee.Number(lat).format('%.5f'))
    return feature.set({'pixel_id': pixel_id})

final_features = sampled_pixels.map(format_pixel_data)

print("Extracting pixel metrics from Earth Engine...")
raw_data = final_features.getInfo()

parsed_records = []
for f in raw_data['features']:
    props = f['properties']
    parsed_records.append({
        'Pixel_ID': props.get('pixel_id'),
        'Longitude': props.get('longitude'),
        'Latitude': props.get('latitude'),
        'Seasonal_AETI_mm': props.get('Seasonal_AETI_mm'),
        'Peak_NDVI': props.get('Peak_NDVI'),
        'Volumetric_Soil_Moisture_Pct': props.get('Volumetric_Soil_Moisture_Pct')
    })

df = pd.DataFrame(parsed_records).dropna(subset=['Seasonal_AETI_mm'])

csv_filename = "wapor_aeti_peakndvi_vsm_pixels.csv"
df.to_csv(csv_filename, index=False)
print(f"Complete! Downloaded {len(df)} pixel records to {csv_filename}")

files.download(csv_filename)

Please upload your KML file:


Saving Chacha cluster2_A=86 ha.kml to Chacha cluster2_A=86 ha (2).kml
KML boundary loaded successfully.
Extracting pixel metrics from Earth Engine...
Complete! Downloaded 89 pixel records to wapor_aeti_peakndvi_vsm_pixels.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================================================================
# STEP 1: Install Dependencies & Initialize Earth Engine
# =====================================================================
!pip install -q earthengine-api pandas fastkml shapely

import ee
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import files
from fastkml import kml
from shapely.geometry import mapping

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-birukgetaneh13')
except Exception:
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project='ee-birukgetaneh13')

# Upload & Parse KML
print("Please upload your KML file:")
uploaded = files.upload()

kml_filename = list(uploaded.keys())[0]
kml_data = uploaded[kml_filename]

def extract_geometries(element):
    geoms = []
    if hasattr(element, 'features'):
        for feat in element.features:
            geoms.extend(extract_geometries(feat))
    if hasattr(element, 'geometry') and element.geometry is not None:
        geoms.append(element.geometry)
    return geoms

# Try standard fastkml parsing first
raw_geoms = []
try:
    k = kml.KML()
    k.from_string(kml_data)
    raw_geoms = extract_geometries(k)
except Exception:
    pass

# Fallback to direct XML parsing if fastkml fails to find geometries
if not raw_geoms:
    try:
        root = ET.fromstring(kml_data)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        coords_elements = root.findall('.//kml:coordinates', namespaces)
        if not coords_elements:
            coords_elements = root.findall('.//coordinates')

        for elem in coords_elements:
            coords_text = elem.text.strip()
            coords_list = []
            for coord_str in coords_text.split():
                parts = coord_str.split(',')
                if len(parts) >= 2:
                    coords_list.append([float(parts[0]), float(parts[1])])
            if coords_list:
                if coords_list[0] == coords_list[-1] and len(coords_list) >= 4:
                    raw_geoms.append(ee.Geometry.Polygon([coords_list]))
                else:
                    raw_geoms.append(ee.Geometry.LineString(coords_list))
    except Exception as e:
        print(f"XML parsing fallback encountered error: {e}")

if not raw_geoms:
    raise ValueError("No geometries could be parsed from the uploaded KML file.")

# Convert geometries safely
if isinstance(raw_geoms[0], ee.Geometry):
    ee_geometries = raw_geoms
else:
    ee_geometries = [ee.Geometry(mapping(g)) for g in raw_geoms]

kml_roi = ee.Geometry.MultiPolygon([g.coordinates() for g in ee_geometries]) if len(ee_geometries) > 1 else ee_geometries[0]
roi_feature = ee.FeatureCollection([ee.Feature(kml_roi)])
print("KML boundary loaded successfully.")

# =====================================================================
# STEP 2: Define Dates & Load Datasets
# =====================================================================
planting_date = '2025-12-24'
harvesting_date = '2026-06-05'

# 1. FAO WaPOR v3 AETI (Sum across season in mm)
aeti_col = (ee.ImageCollection("projects/UNFAO/wapor/v3/L2-AETI-D")
            .filterDate(planting_date, harvesting_date)
            .filterBounds(kml_roi)
            .map(lambda img: img.select('L2-AETI-D').multiply(0.1)
                 .multiply(ee.Date(img.get('system:time_end')).difference(ee.Date(img.get('system:time_start')), 'day')).toFloat()))
seasonal_aeti = aeti_col.sum().rename('Seasonal_AETI_mm').clip(kml_roi)

# 2. Sentinel-2 Peak/Maximum NDVI (10m Resolution)
def add_ndvi(img):
    return img.addBands(img.normalizedDifference(['B8', 'B4']).rename('NDVI'))

s2_ndvi = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(planting_date, harvesting_date)
           .filterBounds(kml_roi)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .map(add_ndvi)
           .select('NDVI')
           .max()  # Captures peak growing season vigor
           .rename('Peak_NDVI')
           .clip(kml_roi))

# 3. Sentinel-1 Volumetric Soil Moisture Conversion (10m Resolution)
s1_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(kml_roi)
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV'))

# Historical reference range for backscatter normalization (last 3 years)
s1_min = s1_col.filterDate('2023-01-01', '2026-06-05').min() # Dry backscatter state
s1_max = s1_col.filterDate('2023-01-01', '2026-06-05').max() # Wet backscatter state

# Current season mean Sentinel-1 VV backscatter
s1_seasonal = s1_col.filterDate(planting_date, harvesting_date).mean()

# Normalized relative index (0 to 1 scale)
ssm_relative = (s1_seasonal.subtract(s1_min)) \
    .divide(s1_max.subtract(s1_min)) \
    .clamp(0, 1)

# Physical Volumetric Soil Moisture (VSM) conversion
# Adjust vsm_min and vsm_max based on your regional soil texture (e.g., Loam/Silt Loam bounds)
vsm_min = 0.10  # 10% Volumetric Water Content (Dry boundary)
vsm_max = 0.42  # 42% Volumetric Water Content (Wet / Field Capacity boundary)

vsm = ssm_relative.multiply(vsm_max - vsm_min).add(vsm_min) \
    .multiply(100) \
    .rename('Volumetric_Soil_Moisture_Pct') \
    .clip(kml_roi)

# =====================================================================
# STEP 3: Combine & Sample Pixel Values for Export
# =====================================================================
coords = ee.Image.pixelLonLat()
combined_composite = ee.Image.cat([
    seasonal_aeti,
    s2_ndvi,
    vsm,
    coords
])

sampled_pixels = combined_composite.sampleRegions(
    collection=roi_feature,
    scale=100,  # Matches WaPOR grid scale
    geometries=False
)

def format_pixel_data(feature):
    lon = feature.get('longitude')
    lat = feature.get('latitude')
    pixel_id = ee.String('Pixel_').cat(ee.Number(lon).format('%.5f')).cat('_').cat(ee.Number(lat).format('%.5f'))
    return feature.set({'pixel_id': pixel_id})

final_features = sampled_pixels.map(format_pixel_data)

print("Extracting pixel metrics from Earth Engine...")
raw_data = final_features.getInfo()

parsed_records = []
for f in raw_data['features']:
    props = f['properties']
    parsed_records.append({
        'Pixel_ID': props.get('pixel_id'),
        'Longitude': props.get('longitude'),
        'Latitude': props.get('latitude'),
        'Seasonal_AETI_mm': props.get('Seasonal_AETI_mm'),
        'Peak_NDVI': props.get('Peak_NDVI'),
        'Volumetric_Soil_Moisture_Pct': props.get('Volumetric_Soil_Moisture_Pct')
    })

df = pd.DataFrame(parsed_records).dropna(subset=['Seasonal_AETI_mm'])

csv_filename = "wapor_aeti_peakndvi_vsm_pixels.csv"
df.to_csv(csv_filename, index=False)
print(f"Complete! Downloaded {len(df)} pixel records to {csv_filename}")

files.download(csv_filename)

Please upload your KML file:


Saving Chacha cluster3_A=31 ha.kml to Chacha cluster3_A=31 ha (1).kml
KML boundary loaded successfully.
Extracting pixel metrics from Earth Engine...
Complete! Downloaded 32 pixel records to wapor_aeti_peakndvi_vsm_pixels.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
##########spatial data in Geotiff#####

for all paramters

In [5]:
# =====================================================================
# STEP 1: Install Dependencies
# =====================================================================
!pip install -q earthengine-api pandas fastkml shapely geemap

import ee
import pandas as pd
import xml.etree.ElementTree as ET
import geemap
from google.colab import files
from fastkml import kml
from shapely.geometry import mapping

# =====================================================================
# STEP 2: Authenticate and Initialize Earth Engine
# =====================================================================
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    # Replace with your GCP Project ID if needed
    ee.Initialize(project='ee-birukgetaneh13')

# =====================================================================
# STEP 3: Upload and Parse KML File
# =====================================================================
print("Please upload your KML file:")
uploaded = files.upload()

kml_filename = list(uploaded.keys())[0]
kml_data = uploaded[kml_filename]

def extract_geometries(element):
    geoms = []
    if hasattr(element, 'features'):
        for feat in element.features:
            geoms.extend(extract_geometries(feat))
    if hasattr(element, 'geometry') and element.geometry is not None:
        geoms.append(element.geometry)
    return geoms

# Try standard fastkml parsing first
raw_geoms = []
try:
    k_obj = kml.KML()
    k_obj.from_string(kml_data)
    raw_geoms = extract_geometries(k_obj)
except Exception:
    pass

# Fallback to direct XML parsing if fastkml fails
if not raw_geoms:
    try:
        root = ET.fromstring(kml_data)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        coords_elements = root.findall('.//kml:coordinates', namespaces)
        if not coords_elements:
            coords_elements = root.findall('.//coordinates')

        for elem in coords_elements:
            coords_text = elem.text.strip()
            coords_list = []
            for coord_str in coords_text.split():
                parts = coord_str.split(',')
                if len(parts) >= 2:
                    coords_list.append([float(parts[0]), float(parts[1])])
            if coords_list:
                if coords_list[0] == coords_list[-1] and len(coords_list) >= 4:
                    raw_geoms.append(ee.Geometry.Polygon([coords_list]))
                else:
                    raw_geoms.append(ee.Geometry.LineString(coords_list))
    except Exception as e:
        print(f"XML parsing fallback encountered error: {e}")

if not raw_geoms:
    raise ValueError("No geometries could be parsed from the uploaded KML file.")

if isinstance(raw_geoms[0], ee.Geometry):
    ee_geometries = raw_geoms
else:
    ee_geometries = [ee.Geometry(mapping(g)) for g in raw_geoms]

if len(ee_geometries) > 1:
    kml_roi = ee.Geometry.MultiPolygon([g.coordinates() for g in ee_geometries])
else:
    kml_roi = ee_geometries[0]

roi_feature = ee.FeatureCollection([ee.Feature(kml_roi)])
print("KML boundary loaded successfully.")

# =====================================================================
# STEP 4: Define Dates & Load Datasets for Wheat Season
# =====================================================================
planting_date = '2025-12-24'
harvesting_date = '2026-06-05'

datasets = {
    'AETI': {'id': 'projects/UNFAO/wapor/v3/L2-AETI-D', 'band': 'L2-AETI-D', 'scale': 0.1},
    'NPP':  {'id': 'projects/UNFAO/wapor/v3/L2-NPP-D',  'band': 'L2-NPP-D',  'scale': 0.001},
    'T':    {'id': 'projects/UNFAO/wapor/v3/L2-T-D',    'band': 'L2-T-D',    'scale': 0.1},
    'ETo':  {'id': 'projects/UNFAO/wapor/v3/L1-RET-D',  'band': 'L1-RET-D',  'scale': 0.1}
}

def load_and_scale_dekads(cfg, var_name):
    def process_img(img):
        start_time = ee.Date(img.get('system:time_start'))
        end_time = ee.Date(img.get('system:time_end'))
        n_days = end_time.difference(start_time, 'day')
        daily_rate = img.select(cfg['band']).multiply(cfg['scale'])
        dekad_total = daily_rate.multiply(n_days).rename(var_name).toFloat()
        return dekad_total

    return (ee.ImageCollection(cfg['id'])
            .filterDate(planting_date, harvesting_date)
            .filterBounds(kml_roi)
            .map(process_img))

aeti_col = load_and_scale_dekads(datasets['AETI'], 'AETI_mm')
npp_col  = load_and_scale_dekads(datasets['NPP'],  'NPP_gC_m2')
t_col    = load_and_scale_dekads(datasets['T'],    'T_mm')
eto_col  = load_and_scale_dekads(datasets['ETo'],  'ETo_mm')

# =====================================================================
# STEP 5: Seasonal Aggregation
# =====================================================================
seasonal_aeti = aeti_col.sum()
seasonal_npp  = npp_col.sum()
seasonal_t    = t_col.sum()
seasonal_eto  = eto_col.sum()
seasonal_biomass = seasonal_npp.multiply(14.443).rename('Total_Biomass')

# =====================================================================
# STEP 6: Visualize on Interactive Map (Using SATELLITE basemap)
# =====================================================================
Map = geemap.Map()
Map.add_basemap('SATELLITE')
Map.centerObject(kml_roi, 11)

Map.addLayer(roi_feature, {'color': 'red'}, 'KML Boundary')
Map.addLayer(
    seasonal_aeti.clip(kml_roi),
    {'min': 100, 'max': 800, 'palette': ['blue', 'green', 'yellow', 'red']},
    'Seasonal AETI (mm)'
)
Map.add_layer_control()
display(Map)

# =====================================================================
# STEP 7: Export Individual Single-Band GeoTIFFs to Google Drive
# =====================================================================
variables = {
    'AETI': seasonal_aeti,
    'ETo': seasonal_eto,
    'Transpiration': seasonal_t,
    'NPP': seasonal_npp,
    'Total_Biomass': seasonal_biomass
}

for var_name, img in variables.items():
    task = ee.batch.Export.image.toDrive(
        image=img.clip(kml_roi),
        description=f'Wheat_Season_{var_name}',
        folder='WaPOR_Exports',          # Target Google Drive folder
        fileNamePrefix=f'wheat_season_{var_name.lower()}',
        region=kml_roi,
        scale=100,                       # 100-meter resolution
        maxPixels=1e9
    )
    task.start()
    print(f"Started export task for single-band raster: {var_name}")

print("\nAll individual raster export tasks initiated! Check your Google Earth Engine Tasks tab and Google Drive 'WaPOR_Exports' folder.")

Please upload your KML file:


Saving Chacha cluster3_A=31 ha.kml to Chacha cluster3_A=31 ha.kml
KML boundary loaded successfully.


Map(center=[9.516803383303474, 39.47931947312385], controls=(WidgetControl(options=['position', 'transparent_b…

Started export task for single-band raster: AETI
Started export task for single-band raster: ETo
Started export task for single-band raster: Transpiration
Started export task for single-band raster: NPP
Started export task for single-band raster: Total_Biomass

All individual raster export tasks initiated! Check your Google Earth Engine Tasks tab and Google Drive 'WaPOR_Exports' folder.


In [8]:
# =====================================================================
# STEP 1: Install Dependencies
# =====================================================================
!pip install -q earthengine-api pandas fastkml shapely geemap

import ee
import pandas as pd
import xml.etree.ElementTree as ET
import geemap
from google.colab import files
from fastkml import kml
from shapely.geometry import mapping

# =====================================================================
# STEP 2: Authenticate and Initialize Earth Engine
# =====================================================================
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    # Replace with your GCP Project ID if needed
    ee.Initialize(project='ee-birukgetaneh13')

# =====================================================================
# STEP 3: Upload and Parse KML File
# =====================================================================
print("Please upload your KML file:")
uploaded = files.upload()

kml_filename = list(uploaded.keys())[0]
kml_data = uploaded[kml_filename]

def extract_geometries(element):
    geoms = []
    if hasattr(element, 'features'):
        for feat in element.features:
            geoms.extend(extract_geometries(feat))
    if hasattr(element, 'geometry') and element.geometry is not None:
        geoms.append(element.geometry)
    return geoms

# Try standard fastkml parsing first
raw_geoms = []
try:
    k_obj = kml.KML()
    k_obj.from_string(kml_data)
    raw_geoms = extract_geometries(k_obj)
except Exception:
    pass

# Fallback to direct XML parsing if fastkml fails
if not raw_geoms:
    try:
        root = ET.fromstring(kml_data)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        coords_elements = root.findall('.//kml:coordinates', namespaces)
        if not coords_elements:
            coords_elements = root.findall('.//coordinates')

        for elem in coords_elements:
            coords_text = elem.text.strip()
            coords_list = []
            for coord_str in coords_text.split():
                parts = coord_str.split(',')
                if len(parts) >= 2:
                    coords_list.append([float(parts[0]), float(parts[1])])
            if coords_list:
                if coords_list[0] == coords_list[-1] and len(coords_list) >= 4:
                    raw_geoms.append(ee.Geometry.Polygon([coords_list]))
                else:
                    raw_geoms.append(ee.Geometry.LineString(coords_list))
    except Exception as e:
        print(f"XML parsing fallback encountered error: {e}")

if not raw_geoms:
    raise ValueError("No geometries could be parsed from the uploaded KML file.")

if isinstance(raw_geoms[0], ee.Geometry):
    ee_geometries = raw_geoms
else:
    ee_geometries = [ee.Geometry(mapping(g)) for g in raw_geoms]

if len(ee_geometries) > 1:
    kml_roi = ee.Geometry.MultiPolygon([g.coordinates() for g in ee_geometries])
else:
    kml_roi = ee_geometries[0]

roi_feature = ee.FeatureCollection([ee.Feature(kml_roi)])
print("KML boundary loaded successfully.")

# =====================================================================
# STEP 4: Define Dates & Load Datasets for Wheat Season
# =====================================================================
planting_date = '2025-12-24'
harvesting_date = '2026-06-05'

datasets = {
    'AETI': {'id': 'projects/UNFAO/wapor/v3/L2-AETI-D', 'band': 'L2-AETI-D', 'scale': 0.1},
    'NPP':  {'id': 'projects/UNFAO/wapor/v3/L2-NPP-D',  'band': 'L2-NPP-D',  'scale': 0.001},
    'T':    {'id': 'projects/UNFAO/wapor/v3/L2-T-D',    'band': 'L2-T-D',    'scale': 0.1},
    'ETo':  {'id': 'projects/UNFAO/wapor/v3/L1-RET-D',  'band': 'L1-RET-D',  'scale': 0.1}
}

def load_and_scale_dekads(cfg, var_name):
    def process_img(img):
        start_time = ee.Date(img.get('system:time_start'))
        end_time = ee.Date(img.get('system:time_end'))
        n_days = end_time.difference(start_time, 'day')
        daily_rate = img.select(cfg['band']).multiply(cfg['scale'])
        dekad_total = daily_rate.multiply(n_days).rename(var_name).toFloat()
        return dekad_total

    return (ee.ImageCollection(cfg['id'])
            .filterDate(planting_date, harvesting_date)
            .filterBounds(kml_roi)
            .map(process_img))

aeti_col = load_and_scale_dekads(datasets['AETI'], 'AETI_mm')
npp_col  = load_and_scale_dekads(datasets['NPP'],  'NPP_gC_m2')
t_col    = load_and_scale_dekads(datasets['T'],    'T_mm')
eto_col  = load_and_scale_dekads(datasets['ETo'],  'ETo_mm')

# =====================================================================
# STEP 5: Seasonal Aggregation & Indicator Calculations
# =====================================================================
seasonal_aeti = aeti_col.sum()
seasonal_npp  = npp_col.sum()
seasonal_t    = t_col.sum()
seasonal_eto  = eto_col.sum()

# 1. Yield = Total Biomass * 0.4 (where Total Biomass = NPP * 14.443)
seasonal_biomass = seasonal_npp.multiply(14.443).rename('Total_Biomass')
yield_raster = seasonal_biomass.multiply(0.4).rename('Yield')

# 2. Water Productivity (WP) = Yield / (AETI * 10)
wp_raster = yield_raster.divide(seasonal_aeti.multiply(10)).rename('WP')

# 3. Irrigation Water Use Efficiency (IWUE) = Yield / [(AETI - 106.3) * 10]
iwue_raster = yield_raster.divide(seasonal_aeti.subtract(106.3).multiply(10)).rename('IWUE')

# 4. Equity = Coefficient of Variation (CV) of AETI across dekads
aeti_mean = aeti_col.reduce(ee.Reducer.mean())
aeti_std  = aeti_col.reduce(ee.Reducer.stdDev())
equity_raster = aeti_std.divide(aeti_mean).rename('Equity_CV')

# 5. Adequacy = AETI / ETc (ETc = 434 mm)
adequacy_raster = seasonal_aeti.divide(434).rename('Adequacy')

# 6. Beneficial Fraction = Transpiration / AETI
beneficial_fraction_raster = seasonal_t.divide(seasonal_aeti).rename('Beneficial_Fraction')

# 7. RWD (Relative Water Deficit) = 1 - Adequacy
rwd_raster = ee.Image(1).subtract(adequacy_raster).rename('RWD')

# =====================================================================
# STEP 6: Visualize AETI on Interactive Map
# =====================================================================
Map = geemap.Map()
Map.add_basemap('SATELLITE')
Map.centerObject(kml_roi, 11)

Map.addLayer(roi_feature, {'color': 'red'}, 'KML Boundary')
Map.addLayer(
    seasonal_aeti.clip(kml_roi),
    {'min': 100, 'max': 800, 'palette': ['blue', 'green', 'yellow', 'red']},
    'Seasonal AETI (mm)'
)
Map.add_layer_control()
display(Map)

# =====================================================================
# STEP 7: Export All Rasters as Individual Single-Band GeoTIFFs
# =====================================================================
variables = {
    'Yield': yield_raster,
    'WP': wp_raster,
    'IWUE': iwue_raster,
    'Equity_CV': equity_raster,
    'Adequacy': adequacy_raster,
    'Beneficial_Fraction': beneficial_fraction_raster,
    'RWD': rwd_raster
}

for var_name, img in variables.items():
    task = ee.batch.Export.image.toDrive(
        image=img.clip(kml_roi),
        description=f'Wheat_Season_{var_name}',
        folder='WaPOR_Exports',          # Target Google Drive folder
        fileNamePrefix=f'wheat_season_{var_name.lower()}',
        region=kml_roi,
        scale=100,                       # 100-meter resolution
        maxPixels=1e9
    )
    task.start()
    print(f"Started export task for single-band raster: {var_name}")

print("\nAll indicator export tasks initiated! Check your Google Earth Engine Tasks tab and Google Drive 'WaPOR_indicators cluster_3' folder.")

Please upload your KML file:


Saving Chacha cluster3_A=31 ha.kml to Chacha cluster3_A=31 ha (1).kml
KML boundary loaded successfully.


Map(center=[9.516803383284675, 39.479319473125344], controls=(WidgetControl(options=['position', 'transparent_…

Started export task for single-band raster: Yield
Started export task for single-band raster: WP
Started export task for single-band raster: IWUE
Started export task for single-band raster: Equity_CV
Started export task for single-band raster: Adequacy
Started export task for single-band raster: Beneficial_Fraction
Started export task for single-band raster: RWD

All indicator export tasks initiated! Check your Google Earth Engine Tasks tab and Google Drive 'WaPOR_indicators cluster_2' folder.


In [11]:
# =====================================================================
# STEP 1: Install Dependencies & Initialize Earth Engine
# =====================================================================
!pip install -q earthengine-api pandas fastkml shapely

import ee
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import files
from fastkml import kml
from shapely.geometry import mapping

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-birukgetaneh13')
except Exception:
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project='ee-birukgetaneh13')

# Upload & Parse KML
print("Please upload your KML file:")
uploaded = files.upload()

kml_filename = list(uploaded.keys())[0]
kml_data = uploaded[kml_filename]

def extract_geometries(element):
    geoms = []
    if hasattr(element, 'features'):
        for feat in element.features:
            geoms.extend(extract_geometries(feat))
    if hasattr(element, 'geometry') and element.geometry is not None:
        geoms.append(element.geometry)
    return geoms

# Try standard fastkml parsing first
raw_geoms = []
try:
    k = kml.KML()
    k.from_string(kml_data)
    raw_geoms = extract_geometries(k)
except Exception:
    pass

# Fallback to direct XML parsing if fastkml fails to find geometries
if not raw_geoms:
    try:
        root = ET.fromstring(kml_data)
        namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}
        coords_elements = root.findall('.//kml:coordinates', namespaces)
        if not coords_elements:
            coords_elements = root.findall('.//coordinates')

        for elem in coords_elements:
            coords_text = elem.text.strip()
            coords_list = []
            for coord_str in coords_text.split():
                parts = coord_str.split(',')
                if len(parts) >= 2:
                    coords_list.append([float(parts[0]), float(parts[1])])
            if coords_list:
                if coords_list[0] == coords_list[-1] and len(coords_list) >= 4:
                    raw_geoms.append(ee.Geometry.Polygon([coords_list]))
                else:
                    raw_geoms.append(ee.Geometry.LineString(coords_list))
    except Exception as e:
        print(f"XML parsing fallback encountered error: {e}")

if not raw_geoms:
    raise ValueError("No geometries could be parsed from the uploaded KML file.")

# Convert geometries safely
if isinstance(raw_geoms[0], ee.Geometry):
    ee_geometries = raw_geoms
else:
    ee_geometries = [ee.Geometry(mapping(g)) for g in raw_geoms]

kml_roi = ee.Geometry.MultiPolygon([g.coordinates() for g in ee_geometries]) if len(ee_geometries) > 1 else ee_geometries[0]
roi_feature = ee.FeatureCollection([ee.Feature(kml_roi)])
print("KML boundary loaded successfully.")

# =====================================================================
# STEP 2: Define Dates & Load Datasets (NDVI & Soil Moisture)
# =====================================================================
planting_date = '2025-12-24'
harvesting_date = '2026-06-05'

# 1. Sentinel-2 Peak/Maximum NDVI (10m Resolution)
def add_ndvi(img):
    return img.addBands(img.normalizedDifference(['B8', 'B4']).rename('NDVI'))

s2_ndvi = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(planting_date, harvesting_date)
           .filterBounds(kml_roi)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .map(add_ndvi)
           .select('NDVI')
           .max()  # Captures peak growing season vigor
           .rename('Peak_NDVI')
           .clip(kml_roi))

# 2. Sentinel-1 Volumetric Soil Moisture Conversion (10m Resolution)
s1_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(kml_roi)
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV'))

# Historical reference range for backscatter normalization
s1_min = s1_col.filterDate('2023-01-01', '2026-06-05').min()
s1_max = s1_col.filterDate('2023-01-01', '2026-06-05').max()

# Current season mean Sentinel-1 VV backscatter
s1_seasonal = s1_col.filterDate(planting_date, harvesting_date).mean()

# Normalized relative index (0 to 1 scale)
ssm_relative = (s1_seasonal.subtract(s1_min)) \
    .divide(s1_max.subtract(s1_min)) \
    .clamp(0, 1)

vsm_min = 0.10  # 10% Volumetric Water Content (Dry boundary)
vsm_max = 0.42  # 42% Volumetric Water Content (Wet / Field Capacity boundary)

vsm = ssm_relative.multiply(vsm_max - vsm_min).add(vsm_min) \
    .multiply(100) \
    .rename('Volumetric_Soil_Moisture_Pct') \
    .clip(kml_roi)

# =====================================================================
# STEP 3: Export Raster GeoTIFFs to Google Drive
# =====================================================================
raster_exports = {
    'Peak_NDVI_c3': s2_ndvi,
    'Volumetric_SM_C3': vsm
}

for var_name, img in raster_exports.items():
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f'Wheat_Season_{var_name}',
        folder='WaPOR_Exports',
        fileNamePrefix=f'wheat_season_{var_name.lower()}',
        region=kml_roi,
        scale=10,                        # 10m native resolution for Sentinel sensors
        maxPixels=1e9
    )
    task.start()
    print(f"Started export task for single-band GeoTIFF: {var_name}")

print("\nRaster export tasks initiated! Check your GEE Tasks tab and Google Drive 'WaPOR_Exports' folder.")

Please upload your KML file:


Saving Chacha cluster3_A=31 ha.kml to Chacha cluster3_A=31 ha (2).kml
KML boundary loaded successfully.
Started export task for single-band GeoTIFF: Peak_NDVI_c2
Started export task for single-band GeoTIFF: Volumetric_SM_C2

Raster export tasks initiated! Check your GEE Tasks tab and Google Drive 'WaPOR_Exports' folder.
